# Prompt extractor

Reads **final_dataset.csv**, adds a **prompt** column (constructed using the same logic as PHASE_1_RUN), and saves **final_dataset_v2.csv**.

- **DS1000**: uses **prompt_2** (and code_context) via `contruct_prompt_ds1k_v4`.
- **HumanEval**: uses prompt via `construct_prompt_humaneval`.
- **MBPP**: uses prompt + function_signature via `construct_prompt_mbpp`.

Column order: `prompt` is placed immediately after `task_id`; `canonical_solution` remains last.

In [1]:
import pandas as pd
import os
import re

FOLDER = os.getcwd()
if not os.path.isfile(os.path.join(FOLDER, "final_dataset.csv")):
    FOLDER = os.path.join(os.getcwd(), "Pipeline construction", "AST+DYNMAIC+LIB_API")
print("Using folder:", FOLDER)

Using folder: d:\Desktop\MIT\CODES\FOR GIT HUB\FYP-26\Pipeline construction\AST+DYNMAIC+LIB_API


## Prompt builders (from PHASE_1_RUN)

In [2]:
def construct_prompt_humaneval(docstring_prompt):
    system_message = (
        "You are an expert Python developer. Your task is to complete the function "
        "provided by the user. Follow the docstring exactly. "
        "Provide your output ONLY as a single Python code block starting with ```python."
    )
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": f"Complete this Python function:\n{docstring_prompt}"}
    ]
    return messages

def construct_prompt_mbpp(prompt_text, signature):
    system_message = (
        "You are an expert Python developer. Your task is to implement a function "
        "based on a description and a specific function signature. "
        "Provide your output ONLY as a single Python code block starting with ```python."
    )
    user_content = (
        f"Problem Description:\n{prompt_text}\n\n"
        f"Please implement this exact function:\n{signature}"
    )
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_content}
    ]
    return messages

def extract_signature(code):
    if pd.isna(code) or not code:
        return ""
    for line in str(code).splitlines():
        line = line.strip()
        if line.startswith("def "):
            return line
    return ""

def extract_only_exec_context_wi(code_context):
    if pd.isna(code_context) or not code_context:
        return "import pandas as pd\nimport numpy as np"
    pattern = r'exec_context = r"""(.*?)"""'
    match = re.search(pattern, str(code_context), re.DOTALL)
    if match:
        return match.group(1).strip()
    return "import pandas as pd\nimport numpy as np"

def contruct_prompt_ds1k_v4(raw_prompt, exec_context_snippet):
    system_message = (
        "You are a Python data scientist.\n"
        "You write concise, correct Python code for data manipulation.\n"
        "You prefer direct, vectorized solutions over complex logic."
    )
    user_message = (
        "You are given a Python code snippet with a placeholder [insert].\n"
        "Your code will be INSERTED at that position.\n\n"
        "===== EXISTING CODE =====\n"
        f"{exec_context_snippet}\n\n"
        "===== TASK =====\n"
        f"{raw_prompt}\n\n"
        "===== GUIDELINES =====\n"
        "- Think deeply about the given TASK before coding.\n"
        "- Write the simplest correct solution.\n"
        "- Prefer short, direct Pandas / NumPy operations.\n"
        "- Do NOT define helper functions or classes.\n"
        "- Do NOT print anything.\n"
        "- You MAY add imports if needed.\n"
        "- Use existing variables from the context.\n"
        "- Make sure `result` variable is declared before using it.\n"
        "- Give proper intendation at [INSERT] if the line before [INSERT] is a function.\n"
        "- Ensure the final output is available in variable `result`.This is really important.\n\n"
        "===== OUTPUT =====\n"
        "Return ONLY raw Python code.\n"
    )
    return [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

def messages_to_prompt_string(messages):
    parts = []
    for m in messages:
        role = m.get("role", "")
        content = m.get("content", "")
        parts.append(f"{role.capitalize()}: {content}")
    return "\n\n".join(parts)

## Load final_dataset.csv and source datasets

In [3]:
from datasets import load_dataset

df_final = pd.read_csv(os.path.join(FOLDER, "final_dataset.csv"))
df_final["task_id"] = df_final["task_id"].astype(str)
print("final_dataset rows:", len(df_final))

print("Loading source datasets...")
ds_ds1k = load_dataset("xlangai/DS-1000")
df_ds1k = ds_ds1k["test"].to_pandas()
df_ds1k["task_id"] = [f"DS{str(i).zfill(4)}" for i in range(len(df_ds1k))]
df_ds1k["prompt_2"] = df_ds1k["prompt"].astype(str).str.split("A:", n=1).str[0].str.strip()

ds_he = load_dataset("openai/openai_humaneval")
df_he = ds_he["test"].to_pandas()

ds_mbpp = load_dataset("google-research-datasets/mbpp", "sanitized")
df_mbpp = ds_mbpp["train"].to_pandas()
df_mbpp["task_id"] = df_mbpp["task_id"].astype(str)
df_mbpp["function_signature"] = df_mbpp["code"].apply(extract_signature)

print("DS1000:", len(df_ds1k), "| HumanEval:", len(df_he), "| MBPP:", len(df_mbpp))

final_dataset rows: 1491
Loading source datasets...


DS1000: 1000 | HumanEval: 164 | MBPP: 120


## Build prompt per row and add column (prompt after task_id)

In [4]:
ds1k_lookup = df_ds1k.set_index("task_id")
he_lookup = df_he.set_index("task_id")
mbpp_lookup = df_mbpp.set_index("task_id")

prompts = []
for _, row in df_final.iterrows():
    dset = row["dataset"]
    tid = str(row["task_id"])
    try:
        if dset == "ds1000":
            r = ds1k_lookup.loc[tid]
            raw_prompt = r.get("prompt_2", r.get("prompt", ""))
            exec_snippet = extract_only_exec_context_wi(r.get("code_context", ""))
            msgs = contruct_prompt_ds1k_v4(raw_prompt, exec_snippet)
        elif dset == "humaneval":
            r = he_lookup.loc[tid]
            msgs = construct_prompt_humaneval(r.get("prompt", ""))
        elif dset == "mbpp":
            r = mbpp_lookup.loc[tid]
            msgs = construct_prompt_mbpp(r.get("prompt", ""), r.get("function_signature", ""))
        else:
            msgs = []
        prompt_str = messages_to_prompt_string(msgs) if msgs else ""
    except Exception:
        prompt_str = ""
    prompts.append(prompt_str)

df_final["prompt"] = prompts

# Place prompt right after task_id; keep canonical_solution last
cols = list(df_final.columns)
cols.remove("prompt")
idx_task = cols.index("task_id")
cols.insert(idx_task + 1, "prompt")
df_final = df_final[cols]
print("Columns (prompt after task_id):", cols)

Columns (prompt after task_id): ['dataset', 'task_id', 'prompt', 'status', 'ast_info', 'dynamic_info', 'lib_info', 'generated_code', 'patched_code', 'error_sources', 'error_types', 'error_lines', 'canonical_solution']


In [5]:
out_path = os.path.join(FOLDER, "final_dataset_v2.csv")
df_final.to_csv(out_path, index=False)
print(f"Saved {len(df_final)} rows to {out_path}")
print("Sample prompt (first 200 chars):", df_final["prompt"].iloc[0][:200] if df_final["prompt"].iloc[0] else "(empty)")

Saved 1491 rows to d:\Desktop\MIT\CODES\FOR GIT HUB\FYP-26\Pipeline construction\AST+DYNMAIC+LIB_API\final_dataset_v2.csv
Sample prompt (first 200 chars): System: You are a Python data scientist.
You write concise, correct Python code for data manipulation.
You prefer direct, vectorized solutions over complex logic.

User: You are given a Python code sn
